# Experiment Template — Backdoor Attack → Detection → Defense

**IMPORTANT RULES**  
- Do NOT change evaluation logic  
- Use shared indices only (`defense_indices.npy`, `asr_test_idx.npy`)  
- Target class = 0 (Airplane)  
- Random seed = 2025  

Each member copies this notebook and ONLY modifies the attack logic section.

In [ ]:
# ============================
# 1. Reproducibility
# ============================
import torch, numpy as np, random

def set_seed(seed=2025):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(2025)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

In [ ]:
# ============================
# 2. Clone Central Repo
# ============================
!git clone https://github.com/Aritraghoshdastidar/adaptive-backdoor-defense.git
%cd adaptive-backdoor-defense

In [ ]:
# ============================
# 3. Imports (DO NOT MODIFY)
# ============================
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

from core.models import get_resnet18
from core.metrics import calculate_ca, calculate_asr
from core.data_utils import load_cifar10


In [ ]:
# ============================
# 4. Load Shared Indices
# ============================
defense_indices = np.load('/content/drive/MyDrive/Shared/defense_indices.npy')
asr_test_idx = np.load('/content/drive/MyDrive/Shared/asr_test_idx.npy')

print('Defense set size:', len(defense_indices))
print('ASR test set size:', len(asr_test_idx))

In [ ]:
# ============================
# 5. Dataset & Dataloaders
# ============================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

trainset = load_cifar10(train=True, transform=transform_train)
testset = load_cifar10(train=False, transform=transform_test)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

## 🔴 MODIFY ONLY THIS SECTION
### Apply your specific attack here (BadNets / Blended / Label-Consistent)

- Poison training data
- Save poisoned indices
- Keep everything else unchanged

In [ ]:
# ============================
# 6. ATTACK LOGIC (EDIT HERE)
# ============================
# TODO: Import your attack from core.attacks
# TODO: Create poisoned training dataset
pass

In [ ]:
# ================================
# Model, Loss, and Optimizer Setup
# ================================

import torch.nn as nn
import torch.optim as optim
from core.models import get_resnet18 # Adjust import based on your template's structure

# 1. Initialize Model
model = get_resnet18().to(device)

# 2. Define Loss Function
criterion = nn.CrossEntropyLoss()

# 3. Define Optimizer (Strictly SGD with these parameters)
optimizer = optim.SGD(
    model.parameters(), 
    lr=0.01, 
    momentum=0.9, 
    weight_decay=5e-4
)

# 4. Define Learning Rate Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=50
)

In [ ]:
# ============================
# 7. Train Model
# ============================
model = get_resnet18().to(device)

from tqdm import tqdm

epochs = 50

for epoch in range(epochs):
    model.train()
    correct = total = running_loss = 0
    
    for imgs, lbls in tqdm(trainloader):
        imgs, lbls = imgs.to(device), lbls.to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Metrics tracking
        running_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)
        
    # Step the scheduler at the end of every epoch
    scheduler.step()
    
    # Standardized output print
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {running_loss/len(trainloader):.4f} | Train Acc: {100*correct/total:.2f}%")

In [ ]:
# ============================
# 8. Evaluation (DO NOT MODIFY)
# ============================
ca = calculate_ca(model, testloader, device)
print('Clean Accuracy:', ca)

# TODO: create poisoned ASR dataloader using asr_test_idx
# asr = calculate_asr(model, asr_loader, target_class=0, device=device)
# print('ASR:', asr)

## Notes
- Log CA & ASR into shared Google Sheet
- Save checkpoint locally (do not push)
- Do NOT alter evaluation logic